# Part 3: Waste Description Classification

We classify a short waste description (for example "empty green glass bottle with contents") into one of 9 categories, using scikit-learn. Files needed next to this notebook: `waste_descriptions.csv`.

**Plan:** clean the text, split it, compare simple models, study the mistakes, add word embeddings, test the final model once, and wrap it in a function.

## Step 1: Loading Libraries

Standard packages, then scikit-learn tools, then Keras for the embedding step.

In [ ]:
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB, ComplementNB
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC
from sklearn.dummy import DummyClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

np.random.seed(42)
tf.random.set_seed(42)

## Step 2: Load Data

In [ ]:
desc = pd.read_csv("waste_descriptions.csv")
print(desc.shape)
desc.head()

**Result:** 5000 rows and 5 columns. Only `description` (input) and `category` (target) are used. The other three columns are fixed per category, so they would leak the answer (see Part 1).

## Step 3: Clean the Text and Remove Duplicates

We lowercase the text, remove odd characters (keeping hyphens), and drop repeated descriptions **before** splitting so the same text cannot land in two sets.

In [ ]:
def clean_text(text):
    text = text.lower().strip()
    text = re.sub(r"[^a-z0-9\- ]+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

data = desc[["description", "category"]].copy()
data["clean"] = data["description"].apply(clean_text)

print("rows before:", len(data))
data = data.drop_duplicates(subset="clean").reset_index(drop=True)
print("rows after removing duplicates:", len(data))

**Result:** 5000 rows become 4939.

## Step 4: Labels and Splits

Category names become numbers, then a stratified 80/10/10 split (train, validation, test).

In [ ]:
LABELS = sorted(data["category"].unique())
label_to_id = {label: i for i, label in enumerate(LABELS)}
data["y"] = data["category"].map(label_to_id)

train_data, hold_data = train_test_split(data, test_size=0.20, stratify=data["y"], random_state=42)
val_data, test_data = train_test_split(hold_data, test_size=0.50, stratify=hold_data["y"], random_state=42)

print("train / validation / test:", len(train_data), len(val_data), len(test_data))
print(LABELS)

**Result:** 3951 train, 494 validation, 494 test. The test set stays untouched until Step 13.

## Step 5: Baseline Models

Every model is a pipeline: TF-IDF turns the text into numbers, then a classifier. Scored on the validation set only. The dummy model always guesses the biggest class.

In [ ]:
X_train, y_train = train_data["clean"], train_data["y"]
X_val, y_val = val_data["clean"], val_data["y"]

def tfidf():
    return TfidfVectorizer(ngram_range=(1, 2))

baseline_models = {
    "Dummy (biggest class)": DummyClassifier(strategy="most_frequent"),
    "Naive Bayes": Pipeline([("tfidf", tfidf()), ("clf", MultinomialNB())]),
    "Logistic regression": Pipeline([("tfidf", tfidf()), ("clf", LogisticRegression(max_iter=2000))]),
    "Random forest": Pipeline([("tfidf", tfidf()), ("clf", RandomForestClassifier(n_estimators=300, random_state=42))]),
}

rows = []
for name, model in baseline_models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_val)
    rows.append({"model": name,
                 "accuracy": round(accuracy_score(y_val, pred), 4),
                 "macro F1": round(f1_score(y_val, pred, average="macro"), 4)})
print(pd.DataFrame(rows).to_string(index=False))

**Result:** Dummy 0.119, Naive Bayes 0.990, logistic regression 0.990, random forest 0.972. Near-perfect scores reflect easy template text, not real-world accuracy.

## Step 6: Look at the Mistakes

Print every validation mistake of the two best models.

In [ ]:
def show_mistakes(model, name):
    pred = model.predict(X_val)
    wrong = pred != y_val.values
    table = val_data[wrong][["description", "category"]].copy()
    table["predicted"] = [LABELS[i] for i in pred[wrong]]
    print(name, "-", int(wrong.sum()), "mistakes")
    print(table.to_string())
    print()

show_mistakes(baseline_models["Naive Bayes"], "Naive Bayes")
show_mistakes(baseline_models["Logistic regression"], "Logistic regression")

**Result:** 5 mistakes each, 3 shared. In 9 of the 10 a material word (glass, paper, plastic) sits in front of an item from another class (pen, marker, battery). Miscellaneous Trash is the true class in 8 of 10.

## Step 7: Why Do Material Words Win?

Count how often some words appear in training and which categories they belong to.

In [ ]:
for word in ["pen", "marker", "battery", "toy", "rug", "glass", "paper", "plastic", "cardboard"]:
    has_word = train_data["clean"].str.contains(rf"\b{word}\b")
    counts = train_data[has_word]["category"].value_counts().to_dict()
    print(word, int(has_word.sum()), counts)

**Result:** Item words (pen 19, marker 15, battery 21, rug 22) are rare but always one class. Material words are common (glass 334, paper 255) and mostly point to their own class, so the models follow them. "toy" is split 21 Plastic against 20 Miscellaneous Trash.

## Step 8: Compare Variants with Cross-Validation

Five-fold cross-validation on train and validation together (4445 descriptions) separates models that look the same on 494 validation rows. The test set is not used.

In [ ]:
X_dev = pd.concat([train_data["clean"], val_data["clean"]])
y_dev = pd.concat([train_data["y"], val_data["y"]])
folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

def tfidf_words():
    return TfidfVectorizer()

variants = {
    "NB, words": Pipeline([("tfidf", tfidf_words()), ("clf", MultinomialNB())]),
    "NB, words+pairs": Pipeline([("tfidf", tfidf()), ("clf", MultinomialNB())]),
    "ComplementNB, words+pairs": Pipeline([("tfidf", tfidf()), ("clf", ComplementNB())]),
    "LR C=1, words": Pipeline([("tfidf", tfidf_words()), ("clf", LogisticRegression(max_iter=2000))]),
    "LR C=1, words+pairs": Pipeline([("tfidf", tfidf()), ("clf", LogisticRegression(max_iter=2000))]),
    "LR C=10, words+pairs": Pipeline([("tfidf", tfidf()), ("clf", LogisticRegression(C=10, max_iter=2000))]),
    "LinearSVC, words+pairs": Pipeline([("tfidf", tfidf()), ("clf", LinearSVC())]),
}

rows = []
for name, model in variants.items():
    scores = cross_val_score(model, X_dev, y_dev, cv=folds, scoring="accuracy")
    rows.append({"variant": name, "mean accuracy": round(scores.mean(), 4), "spread": round(scores.std(), 4)})
print(pd.DataFrame(rows).to_string(index=False))

**Result:** LinearSVC is best (0.9998), then logistic regression with C=10 (0.9984) and ComplementNB (0.9966). Raising C from 1 to 10 lifts logistic regression from 0.9928 to 0.9984, so regularisation was suppressing the rare item words. The best of 7 variants is slightly optimistic.

## Step 9: The Two Best on the Validation Set

In [ ]:
best_two = {
    "LinearSVC": Pipeline([("tfidf", tfidf()), ("clf", LinearSVC())]),
    "Logistic regression C=10": Pipeline([("tfidf", tfidf()), ("clf", LogisticRegression(C=10, max_iter=2000))]),
}
for name, model in best_two.items():
    model.fit(X_train, y_train)
    print(name, "validation accuracy:", round(accuracy_score(y_val, model.predict(X_val)), 4))
    show_mistakes(model, name)

**Result:** LinearSVC makes 0 mistakes; logistic regression (C=10) makes 3, all of them decoys ("marker", "pen"). Zero errors on 494 rows is not a zero error rate.

## Step 10: Stress Test on Unusual Descriptions

LinearSVC gives no probabilities, so we wrap it in `CalibratedClassifierCV`. The test inputs are my own, including decoys, natural phrases, and edge cases.

In [ ]:
calibrated = Pipeline([("tfidf", tfidf()), ("clf", CalibratedClassifierCV(LinearSVC(), cv=5))])
calibrated.fit(X_train, y_train)

stress_inputs = [
    ("blue glass pen", "Miscellaneous Trash"), ("green plastic marker", "Miscellaneous Trash"),
    ("dry paper battery", "Miscellaneous Trash"), ("old cardboard rug", "Textile Trash"),
    ("white plastic pen", "Miscellaneous Trash"),
    ("an empty glass jar", "Glass"), ("banana peel", "Food Organics"),
    ("grass clippings from the garden", "Vegetation"), ("cotton t-shirt", "Textile Trash"),
    ("aluminium can", "Metal"), ("used battery", "Miscellaneous Trash"), ("newspaper", "Paper"),
    ("", "?"), ("glass", "Glass"), ("asdf qwerty", "?"), ("chupa ya plastiki", "Plastic"),
]

rows = []
for text, expected in stress_inputs:
    probs = calibrated.predict_proba([clean_text(text)])[0]
    best = probs.argmax()
    rows.append({"input": text, "expected": expected, "predicted": LABELS[best], "confidence": round(probs[best], 2)})
print(pd.DataFrame(rows).to_string(index=False))

**Result:** 13 of the 14 answerable inputs are right, including all 5 decoys (the two "pen" decoys at only 0.51). Unknown text (empty, gibberish, Swahili) gets 0.25, below every correct answer, so a confidence threshold can reject it.

## Step 11: Word Embeddings

The brief asks for word embeddings. A small Keras model learns one 32-number vector per word while it learns the categories. These embeddings are learned from the labels, so they are not like Word2Vec.

In [ ]:
max_len = int(train_data["clean"].str.split().str.len().max())

vectorizer = layers.TextVectorization(standardize=None, split="whitespace",
                                      output_mode="int", output_sequence_length=max_len)
vectorizer.adapt(train_data["clean"].values)          # vocabulary from training text only
vocab = vectorizer.get_vocabulary()

X_train_ids = vectorizer(tf.constant(train_data["clean"].values)).numpy()
X_val_ids = vectorizer(tf.constant(val_data["clean"].values)).numpy()

print("longest description (words):", max_len)
print("vocabulary size:", len(vocab))
print("training ids shape:", X_train_ids.shape)

In [ ]:
emb_model = keras.Sequential([
    keras.Input(shape=(max_len,)),
    layers.Embedding(input_dim=len(vocab), output_dim=32, mask_zero=True, name="emb"),
    layers.GlobalAveragePooling1D(),
    layers.Dropout(0.3),
    layers.Dense(len(LABELS), activation="softmax"),
])
emb_model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])

early_stop = keras.callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)
history = emb_model.fit(X_train_ids, y_train, validation_data=(X_val_ids, y_val),
                        epochs=100, batch_size=32, callbacks=[early_stop], verbose=0)

val_loss, val_acc = emb_model.evaluate(X_val_ids, y_val, verbose=0)
print("epochs run:", len(history.history["loss"]))
print("validation loss:", round(val_loss, 5), "| validation accuracy:", round(val_acc, 4))

**Result:** Vocabulary of 307 entries, longest description 10 words. After the full 100 epochs validation accuracy is 1.0 (loss 0.00145), tying LinearSVC.

In [ ]:
word_vectors = emb_model.get_layer("emb").get_weights()[0]
words = [str(w) for w in vocab]
unit_vectors = word_vectors / (np.linalg.norm(word_vectors, axis=1, keepdims=True) + 1e-9)

def nearest_words(word, how_many=6):
    i = words.index(word)
    similarity = unit_vectors @ unit_vectors[i]
    order = np.argsort(-similarity)
    result = []
    for j in order:
        if j > 1 and j != i:                      # skip padding, [UNK] and the word itself
            result.append((words[j], round(float(similarity[j]), 2)))
        if len(result) == how_many:
            break
    return result

for word in ["glass", "plastic", "paper", "pen", "marker", "bottle"]:
    print(word, "->", nearest_words(word))

**Result:** Neighbours group by category, not by meaning: glass sits with wine, beer, perfume; plastic with yogurt, milk, jug; pen and marker with other Miscellaneous Trash items (similarity 0.99 to 1.0). Closeness means "the classifier treats them alike".

In [ ]:
def average_embedding(ids):
    mask = (ids > 0)[:, :, None]                  # real words only (0 = padding)
    total = (word_vectors[ids] * mask).sum(axis=1)
    count = np.maximum(mask.sum(axis=1), 1)       # avoid dividing by zero
    return total / count

E_train = average_embedding(X_train_ids)
E_val = average_embedding(X_val_ids)
print(E_train.shape, E_val.shape)

embedding_models = {
    "Embeddings + logistic regression": LogisticRegression(max_iter=2000),
    "Embeddings + LinearSVC": LinearSVC(),
    "Embeddings + random forest": RandomForestClassifier(n_estimators=300, random_state=42),
}
rows = []
for name, model in embedding_models.items():
    model.fit(E_train, y_train)
    pred = model.predict(E_val)
    rows.append({"model": name, "accuracy": round(accuracy_score(y_val, pred), 4)})
print(pd.DataFrame(rows).to_string(index=False))

**Result:** The 32-number averages give 1.000, 1.000 and 0.998, matching TF-IDF with far fewer numbers. The embeddings were learned from the labels, so this is not independent proof.

In [ ]:
empty_ids = vectorizer(tf.constant([""])).numpy()
print(emb_model.predict(empty_ids, verbose=0))

**Result:** An empty description gives `nan` instead of probabilities (every position is padding, so the average divides by zero). The final prediction function therefore checks for empty and unknown input itself.

## Step 12: Final Model and One Test Evaluation

The final choice was made from cross-validation before touching the test set: TF-IDF, word pairs, calibrated LinearSVC, refit on train and validation together.

In [ ]:
text_model = Pipeline([("tfidf", tfidf()), ("clf", CalibratedClassifierCV(LinearSVC(), cv=5))])
text_model.fit(X_dev, y_dev)

X_test, y_test = test_data["clean"], test_data["y"]
test_pred = text_model.predict(X_test)
print("test accuracy:", round(accuracy_score(y_test, test_pred), 4))

cm = confusion_matrix(y_test, test_pred)
plt.figure(figsize=(8, 7))
plt.imshow(cm, cmap="Blues")
plt.xticks(range(len(LABELS)), LABELS, rotation=45, ha="right")
plt.yticks(range(len(LABELS)), LABELS)
for i in range(len(LABELS)):
    for j in range(len(LABELS)):
        plt.text(j, i, cm[i, j], ha="center", va="center", color="white" if cm[i, j] > cm.max() / 2 else "black")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Confusion matrix (test set)")
plt.tight_layout()
plt.show()

print(classification_report(y_test, test_pred, target_names=LABELS))

**Result:** Test accuracy 1.0 with 0 errors on 494 descriptions, every precision and recall 1.00. This reflects the template-generated data, not real-world performance; the mistakes studied in Steps 6 and 9 and the stress test are the real analysis.

## Step 13: Prediction Function

A function that returns the category, a confidence and a status, and refuses empty or unknown input instead of guessing. The 0.40 threshold lies between the 0.25 given to unknown text and the lowest correct confidence.

In [ ]:
CONFIDENCE_THRESHOLD = 0.40

def predict_waste_category(text, threshold=CONFIDENCE_THRESHOLD):
    if not isinstance(text, str) or clean_text(text) == "":
        return {"category": None, "confidence": 0.0, "status": "empty input", "top3": []}

    cleaned = clean_text(text)
    word_numbers = text_model.named_steps["tfidf"].transform([cleaned])
    if word_numbers.nnz == 0:
        return {"category": None, "confidence": 0.0, "status": "no known words", "top3": []}

    probs = text_model.predict_proba([cleaned])[0]
    order = np.argsort(-probs)[:3]
    top3 = [(LABELS[i], round(float(probs[i]), 2)) for i in order]
    status = "ok" if probs[order[0]] >= threshold else "low confidence"
    return {"category": LABELS[order[0]], "confidence": round(float(probs[order[0]]), 2),
            "status": status, "top3": top3}

for text in ["blue glass pen", "banana peel", "Dry GLASS bottle with contents", "",
             "   ", None, 123, "!!!", "asdf qwerty", "chupa ya plastiki"]:
    print(repr(text), "->", predict_waste_category(text))

**Result:** Four inputs get a category; empty, whitespace, `None`, a number, punctuation only, and text with no known words all return `None` with a reason. With the final model the lowest correct confidence is 0.52, only 0.12 above the threshold, so the threshold should be re-checked whenever the model changes.

## Summary

- **Final model:** TF-IDF (words and word pairs) with a calibrated LinearSVC. Test accuracy 1.0, 0 errors.
- **What the mistakes taught:** a frequent material word (glass, paper) outweighs a rare item word (pen, marker). Loosening the regularisation fixed it.
- **Limitations:** the descriptions are template-generated (307-word vocabulary), so near-perfect accuracy reflects the data. The model cannot read words it never saw, including other languages. The stress test has only 16 hand-written inputs, and the 0.40 threshold is tentative.